# Analyze Tracks and People

Inspect a completed observation and identity-resolution run without rerunning YOLO or InsightFace. This notebook shows run health, track/person statistics, identity switches, timelines, representative faces, and an interactive annotated-frame viewer.

## 1. Configuration

Set `RUN_DIRECTORY` to the run produced by notebooks 01 and 02.

In [ ]:
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
RUN_DIRECTORY = PROJECT_ROOT / "runs" / "source04-full"
MAX_FACES_PER_PERSON = 20
FACES_PER_ROW = 10

RUN_DIRECTORY = Path(RUN_DIRECTORY).expanduser().resolve()
print(f"Project root: {PROJECT_ROOT}")
print(f"Run directory: {RUN_DIRECTORY}")

## 2. Load the resolved run

In [ ]:
import json
import sys
from collections import Counter, defaultdict

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Video, display

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from person_tracker.io import read_nth_frame
from person_tracker.storage import load_observation_run

required_files = [
    "manifest.json", "tracks.jsonl", "face_samples.jsonl",
    "face_embeddings.npy", "identities.jsonl",
]
missing_files = [name for name in required_files if not (RUN_DIRECTORY / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Run is incomplete; missing: {missing_files}")

manifest, tracking_history, face_samples = load_observation_run(
    RUN_DIRECTORY, load_face_crops=True
)

identity_history = defaultdict(dict)
with (RUN_DIRECTORY / "identities.jsonl").open(encoding="utf-8") as stream:
    for line in stream:
        if not line.strip():
            continue
        record = json.loads(line)
        frame_no = int(record.pop("frame"))
        track_id = int(record.pop("track_id"))
        identity_history[frame_no][track_id] = record
identity_history = dict(identity_history)

summary_path = RUN_DIRECTORY / "identity_summary.json"
events_path = RUN_DIRECTORY / "identity_events.json"
identity_summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
switch_events = json.loads(events_path.read_text()) if events_path.exists() else {}

INPUT_VIDEO = Path(manifest["source_video"])
fps = float(manifest["fps"])
start_frame = int(manifest["start_frame"])
end_frame = int(manifest["end_frame"])

print(f"Source: {INPUT_VIDEO}")
print(f"Frames: {start_frame}:{end_frame} ({(end_frame-start_frame)/fps:.2f}s)")
print(f"Tracked frames: {len(tracking_history):,}")
print(f"Face samples: {len(face_samples):,}")
print(f"Resolved people: {identity_summary.get('person_count', 'unknown')}")

## 3. Track and person statistics

In [ ]:
track_stats = defaultdict(lambda: {
    "frames": [], "confidences": [], "people": Counter(), "sources": Counter()
})
person_stats = defaultdict(lambda: {
    "frames": set(), "tracks": set(), "confidences": [], "sources": Counter()
})

for frame_no, tracks in tracking_history.items():
    identities = identity_history.get(frame_no, {})
    for track in tracks:
        track_id = int(track["track_id"])
        stats = track_stats[track_id]
        stats["frames"].append(frame_no)
        stats["confidences"].append(float(track["confidence"]))
        identity = identities.get(track_id)
        if identity:
            person_id = int(identity["person_id"])
            stats["people"][person_id] += 1
            stats["sources"][identity.get("source", "unknown")] += 1
            pstats = person_stats[person_id]
            pstats["frames"].add(frame_no)
            pstats["tracks"].add(track_id)
            pstats["confidences"].append(float(identity.get("confidence", 0.0)))
            pstats["sources"][identity.get("source", "unknown")] += 1

faces_by_track = Counter(sample.track_id for sample in face_samples)
track_rows = []
for track_id, stats in sorted(track_stats.items()):
    frames = stats["frames"]
    track_rows.append({
        "Track": track_id,
        "First frame": min(frames),
        "Last frame": max(frames),
        "Visible frames": len(frames),
        "Duration (s)": len(frames) / fps,
        "Mean detection": np.mean(stats["confidences"]),
        "Resolved people": ", ".join(f"P{p} ({n})" for p, n in stats["people"].most_common()),
        "Face samples": faces_by_track[track_id],
        "Identity sources": dict(stats["sources"]),
    })
track_table = pd.DataFrame(track_rows)
display(HTML("<h3>Tracks</h3>"), track_table.style.format({
    "Duration (s)": "{:.2f}", "Mean detection": "{:.3f}"
}))

In [ ]:
face_person = {}
faces_by_person = defaultdict(list)
for sample in face_samples:
    identity = identity_history.get(sample.frame_no, {}).get(sample.track_id)
    if identity:
        person_id = int(identity["person_id"])
        face_person[id(sample)] = person_id
        faces_by_person[person_id].append(sample)

person_rows = []
for person_id, stats in sorted(person_stats.items()):
    frames = stats["frames"]
    person_rows.append({
        "Person": person_id,
        "First frame": min(frames),
        "Last frame": max(frames),
        "Visible frames": len(frames),
        "Duration (s)": len(frames) / fps,
        "Tracks": sorted(stats["tracks"]),
        "Track count": len(stats["tracks"]),
        "Face samples": len(faces_by_person.get(person_id, [])),
        "Mean identity confidence": np.mean(stats["confidences"]),
        "Identity sources": dict(stats["sources"]),
    })
person_table = pd.DataFrame(person_rows)
display(HTML("<h3>People</h3>"), person_table.style.format({
    "Duration (s)": "{:.2f}", "Mean identity confidence": "{:.3f}"
}))

## 4. Person timeline and identity-switch events

In [ ]:
fig, ax = plt.subplots(figsize=(15, max(4, 0.55 * len(person_stats))))
for person_id in sorted(person_stats):
    frames = sorted(person_stats[person_id]["frames"])
    times = [(frame - start_frame) / fps for frame in frames]
    ax.scatter(times, [person_id] * len(times), s=8, label=f"P{person_id}")
ax.set(title="Resolved person visibility", xlabel="Seconds from run start", ylabel="Person ID")
ax.set_yticks(sorted(person_stats))
ax.grid(True, axis="x", alpha=0.25)
plt.show()

event_rows = []
for track_id, events in switch_events.items():
    for event in events:
        event_rows.append({"Track": int(track_id), **event})
event_table = pd.DataFrame(event_rows)
display(HTML("<h3>Detected identity-switch boundaries</h3>"))
display(event_table if not event_table.empty else HTML("<i>No switch events recorded.</i>"))

## 5. Representative face galleries

In [ ]:
def select_representative_faces(samples, max_faces=20):
    samples = sorted(samples, key=lambda sample: sample.frame_no)
    if len(samples) <= max_faces:
        return samples
    selected = []
    for index in range(max_faces):
        left = round(index * len(samples) / max_faces)
        right = round((index + 1) * len(samples) / max_faces)
        bucket = samples[left:right]
        if bucket:
            selected.append(max(bucket, key=lambda sample: sample.quality))
    return selected

for person_id in sorted(faces_by_person):
    all_samples = faces_by_person[person_id]
    samples = select_representative_faces(all_samples, MAX_FACES_PER_PERSON)
    rows = max(1, int(np.ceil(len(samples) / FACES_PER_ROW)))
    fig, axes = plt.subplots(rows, FACES_PER_ROW, figsize=(FACES_PER_ROW * 1.8, rows * 2.2), squeeze=False)
    tracks = sorted({sample.track_id for sample in all_samples})
    fig.suptitle(f"Person {person_id} | {len(all_samples)} faces | tracks {tracks}", fontsize=14)
    for index, sample in enumerate(samples):
        ax = axes[index // FACES_PER_ROW][index % FACES_PER_ROW]
        if sample.image.size:
            ax.imshow(cv2.cvtColor(sample.image, cv2.COLOR_BGR2RGB))
        ax.set_title(f"T{sample.track_id} F{sample.frame_no}\nQ {sample.quality:.2f}", fontsize=8)
        ax.axis("off")
    for index in range(len(samples), rows * FACES_PER_ROW):
        axes[index // FACES_PER_ROW][index % FACES_PER_ROW].axis("off")
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

## 6. Select the person to render

Clicking a card saves the choice inside this run. Notebook 04 loads it automatically unless its `TARGET_PERSON_ID` is explicitly set.

In [ ]:
from person_tracker.ui import create_person_selection_widget

SELECTION_PATH = RUN_DIRECTORY / "selected_person.json"
selector, selection_state = create_person_selection_widget(
    faces_by_person, person_stats, selection_path=SELECTION_PATH,
)
display(selector)
print("Click a person card to save the selection for notebook 04.")

## 7. Interactive annotated-frame viewer

Move through the analyzed range and optionally isolate one resolved person. Rerun this cell after reopening the notebook to reactivate the controls.

In [ ]:
person_options = [("All people", 0)] + [(f"Person {pid}", pid) for pid in sorted(person_stats)]

def show_annotated_frame(frame_no, person_id):
    frame = read_nth_frame(INPUT_VIDEO, frame_no)
    identities = identity_history.get(frame_no, {})
    shown = 0
    for track in tracking_history.get(frame_no, []):
        track_id = int(track["track_id"])
        identity = identities.get(track_id)
        resolved_person = int(identity["person_id"]) if identity else None
        if person_id and resolved_person != person_id:
            continue
        x1, y1, x2, y2 = map(int, track["bbox"])
        color = (0, 255, 0) if identity else (0, 0, 255)
        label = f"T{track_id} | P{resolved_person if resolved_person is not None else '--'}"
        if identity:
            label += f" | {float(identity.get('confidence', 0)):.0%}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 5)
        cv2.putText(frame, label, (x1, max(35, y1 - 12)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)
        shown += 1
    plt.figure(figsize=(16, 9))
    plt.imshow(frame)
    plt.title(f"Frame {frame_no} | {(frame_no-start_frame)/fps:.2f}s | boxes shown: {shown}")
    plt.axis("off")
    plt.show()

frame_slider = widgets.IntSlider(
    value=start_frame, min=start_frame, max=max(start_frame, end_frame - 1),
    step=1, description="Frame", continuous_update=False,
    layout=widgets.Layout(width="80%"),
)
person_dropdown = widgets.Dropdown(options=person_options, value=0, description="Show")
viewer = widgets.interactive_output(
    show_annotated_frame, {"frame_no": frame_slider, "person_id": person_dropdown}
)
display(widgets.VBox([person_dropdown, frame_slider]), viewer)

## 8. Existing rendered outputs

In [ ]:
candidate_videos = [
    PROJECT_ROOT / "output" / "phase1_tracked.mp4",
    PROJECT_ROOT / "output" / "phase2_target.mp4",
    PROJECT_ROOT / "output" / "final_cropped.mp4",
]
for video_path in candidate_videos:
    if video_path.exists():
        display(HTML(f"<h3>{video_path.name}</h3>"))
        display(Video(filename=str(video_path), embed=False, width=1000))